# Lecture 8 — Bayesian Regression: Treating Uncertainty as a Feature

**Practices covered:** L01, L02 (same code — different conceptual lens), L08 (Regularization)

---

## The Problem with Standard Regression

In Lecture 7, gradient descent gives you **one answer**: "the slope is 1.2".

But with only a little data, how **confident** are you in that answer?

- With 5 data points, the slope could be anywhere from 0.8 to 1.6
- With 500 points, you're much more certain it's close to 1.2

**Bayesian regression** tracks this uncertainty explicitly — instead of a single weight vector, you maintain a **distribution** over all possible weight vectors.

---

## The Core Idea: Bayes' Theorem

$$\underbrace{P(W | \text{data})}_{\text{posterior}} = \frac{\underbrace{P(\text{data} | W)}_{\text{likelihood}} \cdot \underbrace{P(W)}_{\text{prior}}}{\underbrace{P(\text{data})}_{\text{normalizer}}}$$

In plain language:
- **Prior** $P(W)$: what do I believe about the weights *before* seeing data? (e.g., "weights are probably small")
- **Likelihood** $P(\text{data}|W)$: how probable is this data if the weights were $W$?
- **Posterior** $P(W|\text{data})$: what should I believe *after* seeing data?

The posterior is what we want — it tells us *which weight values are plausible given the evidence*.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
%matplotlib inline

# Illustrate: prior → posterior update with 1D example
# Suppose we're estimating the true mean μ of a distribution
# Prior: we think μ is around 0, not too certain
# After seeing data x = [1.2, 0.8, 1.5, 1.1], we update our belief

mu_vals = np.linspace(-3, 5, 300)

# Prior: Gaussian centered at 0, std=2 (not very informative)
prior = norm.pdf(mu_vals, loc=0, scale=2)

# Likelihood of data x=[1.2, 0.8, 1.5, 1.1] given each possible mu
data = np.array([1.2, 0.8, 1.5, 1.1])
likelihood = np.ones_like(mu_vals)
for xi in data:
    likelihood *= norm.pdf(xi, loc=mu_vals, scale=0.5)

# Posterior = prior × likelihood (normalized)
posterior_unnorm = prior * likelihood
posterior = posterior_unnorm / np.trapz(posterior_unnorm, mu_vals)  # normalize

plt.figure(figsize=(8, 4))
plt.plot(mu_vals, prior / prior.max(), 'b--', label='Prior (before data)')
plt.plot(mu_vals, likelihood / likelihood.max(), 'g:', label='Likelihood (data signal)')
plt.plot(mu_vals, posterior / posterior.max(), 'r-', linewidth=2, label='Posterior (after data)')
plt.axvline(data.mean(), color='gray', linestyle=':', label=f'Data mean={data.mean():.2f}')
plt.xlabel('μ'); plt.ylabel('Probability (normalized)')
plt.title('Bayes: Prior × Likelihood → Posterior')
plt.legend(); plt.show()

## Part 2 — Why Normal Distributions?

The lecture uses **Gaussian (Normal) distributions** for both the prior and the noise. Why?

1. **Mathematical convenience:** Gaussian prior × Gaussian likelihood = Gaussian posterior. No messy integrals.
2. **Realistic:** The Central Limit Theorem says measurement noise is often Gaussian.
3. **Conjugate prior:** When prior and posterior are the same family, called *conjugate*. Makes sequential updates trivial.

**The regression model:**
- Observations: $t_n = W^T \phi(x_n) + \epsilon_n$ where $\epsilon_n \sim \mathcal{N}(0, \sigma^2)$
- Prior on weights: $p(W) = \mathcal{N}(W | m_0, S_0)$
- Result: posterior $p(W|\mathbf{t}) = \mathcal{N}(W | m_N, S_N)$ — still Gaussian!

In [ ]:
# Demonstrate sequential updating:
# As we see more data, the posterior gets narrower (more certain)

np.random.seed(42)
true_w = 1.5   # true slope
noise_std = 0.5

# Generate 50 data points sequentially
x_seq = np.linspace(0, 3, 50)
y_seq = true_w * x_seq + noise_std * np.random.randn(50)

# Bayesian update for slope w (assuming known bias=0, known noise)
w_vals = np.linspace(0, 3, 300)

# Prior: broad Gaussian
prior_mu, prior_sigma = 0.0, 5.0
posteriors = []
labels = []

for n_obs in [0, 1, 3, 10, 50]:
    if n_obs == 0:
        pdf = norm.pdf(w_vals, prior_mu, prior_sigma)
    else:
        # Bayesian update formula for 1D slope
        x_n = x_seq[:n_obs]
        y_n = y_seq[:n_obs]
        precision_prior = 1 / prior_sigma**2
        precision_likelihood = np.sum(x_n**2) / noise_std**2
        precision_post = precision_prior + precision_likelihood
        mu_post = (precision_prior*prior_mu + np.sum(x_n*y_n)/noise_std**2) / precision_post
        sigma_post = np.sqrt(1/precision_post)
        pdf = norm.pdf(w_vals, mu_post, sigma_post)
    posteriors.append(pdf / pdf.max())
    labels.append(f'n={n_obs}')

plt.figure(figsize=(8, 4))
for pdf, label in zip(posteriors, labels):
    plt.plot(w_vals, pdf, label=label)
plt.axvline(true_w, color='black', linestyle='--', label='True w=1.5')
plt.xlabel('Slope w'); plt.ylabel('Posterior density (normalized)')
plt.title('Posterior narrows as we see more data')
plt.legend(); plt.show()

## Part 3 — The Key Insight: MAP = Regularized Regression

**Maximum A Posteriori (MAP)** estimation: pick the weight that maximizes the posterior.

$$W_{\text{MAP}} = \arg\max_W \; P(W | \text{data}) = \arg\max_W \; P(\text{data}|W) \cdot P(W)$$

Taking the log (log turns products into sums):

$$W_{\text{MAP}} = \arg\max_W \; \log P(\text{data}|W) + \log P(W)$$

If the **prior is Gaussian** $P(W) = \mathcal{N}(0, \sigma_W^2)$:

$$\log P(W) \propto -\frac{1}{2\sigma_W^2} \|W\|^2$$

Combined with MSE (Gaussian likelihood):

$$W_{\text{MAP}} = \arg\min_W \; \underbrace{\frac{1}{2m}\sum(\hat{y}^{(i)} - y^{(i)})^2}_{\text{data fit}} + \underbrace{\frac{\lambda}{2m}\|W\|^2}_{\text{regularization}}$$

**This is exactly the regularized regression cost from Lecture 7!**

$$\lambda = \frac{\sigma^2}{\sigma_W^2 \cdot m}$$

So **regularization is not just a trick to prevent overfitting — it's the mathematical consequence of having a Gaussian prior belief that weights should be small**.

In [ ]:
# Visualize: MAP solution = minimum of (data loss + prior penalty)
np.random.seed(1)
x_demo = np.array([1, 2, 3, 4, 5], dtype=float)
y_demo = 0.5*x_demo + 0.8*np.random.randn(5)
X_demo = np.column_stack((np.ones(5), x_demo))
Y_demo = y_demo.reshape(-1,1)

w_range = np.linspace(-1, 2, 200)
w0_fixed = 0.0  # fix bias for visualization

# Data fit loss for each w1 value
data_loss = np.array([
    float((1/(2*5)) * (X_demo @ np.array([[w0_fixed],[w]]) - Y_demo).T 
          @ (X_demo @ np.array([[w0_fixed],[w]]) - Y_demo))
    for w in w_range
])

# Prior penalty for each w1 value (lambda=1)
lam = 1.0
prior_penalty = (lam / (2*5)) * w_range**2

# MAP objective = data loss + prior penalty
map_objective = data_loss + prior_penalty

plt.figure(figsize=(8, 4))
plt.plot(w_range, data_loss,     'b-',  label='Data fit (MLE)')
plt.plot(w_range, prior_penalty, 'g--', label=f'Prior penalty (λ={lam})')
plt.plot(w_range, map_objective, 'r-', linewidth=2, label='MAP objective = MLE + Prior')
plt.axvline(w_range[np.argmin(data_loss)],     color='blue',  linestyle=':', label='MLE minimum')
plt.axvline(w_range[np.argmin(map_objective)], color='red',   linestyle=':', label='MAP minimum')
plt.xlabel('w₁ (slope)'); plt.ylabel('Cost')
plt.title('MAP = MLE + Prior: regularization shrinks weights toward 0')
plt.legend(); plt.show()

## Part 4 — Predictive Uncertainty: Confidence Intervals

The main practical advantage of Bayesian regression over plain regression: you get **confidence intervals** on your predictions.

For a new input $x^*$, the prediction is not a single number but a distribution:

$$p(t^* | x^*, \text{data}) = \mathcal{N}(t^* \; | \; m_N^T \phi(x^*), \; \sigma^2_N(x^*))$$

- The **mean** $m_N^T \phi(x^*)$ is the best estimate
- The **variance** $\sigma^2_N(x^*)$ tells you how uncertain you are
- Uncertainty is higher far from training data — this is what plain regression can't tell you!

In [ ]:
# Bayesian regression: show prediction uncertainty bands
from numpy.linalg import inv

np.random.seed(42)
n_train = 10
x_train = np.random.uniform(-1, 1, n_train)
y_train = np.sin(2*x_train) + 0.2*np.random.randn(n_train)

# Design matrix with polynomial features (degree 3)
def phi(x, degree=3):
    return np.column_stack([x**i for i in range(degree+1)])

Phi_train = phi(x_train)
beta = 25.0   # noise precision = 1/noise_variance
alpha = 2.0   # prior precision = 1/prior_variance

# Posterior parameters
S0_inv = alpha * np.eye(4)            # prior precision matrix
SN_inv = S0_inv + beta * Phi_train.T @ Phi_train
SN = inv(SN_inv)
mN = beta * SN @ Phi_train.T @ y_train.reshape(-1,1)

# Predict on test grid
x_test = np.linspace(-1.5, 1.5, 200)
Phi_test = phi(x_test)
pred_mean = (Phi_test @ mN).flatten()
pred_var = np.array([1/beta + float(Phi_test[i] @ SN @ Phi_test[i]) for i in range(len(x_test))])
pred_std = np.sqrt(pred_var)

plt.figure(figsize=(9, 5))
plt.scatter(x_train, y_train, c='black', s=50, zorder=5, label='Training data')
plt.plot(x_test, np.sin(2*x_test), 'g--', label='True function')
plt.plot(x_test, pred_mean, 'b-', linewidth=2, label='Posterior mean')
plt.fill_between(x_test, pred_mean-pred_std, pred_mean+pred_std, alpha=0.2, color='blue', label='±1σ confidence')
plt.fill_between(x_test, pred_mean-2*pred_std, pred_mean+2*pred_std, alpha=0.1, color='blue', label='±2σ confidence')
plt.axvspan(-1.5, -1, alpha=0.05, color='red', label='Extrapolation zone')
plt.axvspan(1, 1.5, alpha=0.05, color='red')
plt.xlabel('x'); plt.ylabel('y')
plt.title('Bayesian prediction: uncertainty is higher far from training data')
plt.legend(loc='upper right'); plt.ylim(-2.5, 2.5); plt.show()

## Part 5 — Connection to the Exam Notebook (L08 Regularization)

Practice L08 explores **Ridge and Lasso** regularization. Now you understand why:

| Regularization | Prior | Effect |
|---|---|---|
| Ridge (L2): $\lambda \|W\|^2$ | Gaussian prior $\mathcal{N}(0, \sigma^2)$ | Shrinks all weights toward 0 smoothly |
| Lasso (L1): $\lambda \|W\|_1$ | Laplace prior | Shrinks many weights to exactly 0 (feature selection) |
| No regularization | Flat prior ("anything is equally likely") | Maximum likelihood — can overfit |

The $\lambda$ hyperparameter controls **how strongly you believe weights should be small**.

In [ ]:
# Lambda sweep: demonstrate bias-variance tradeoff
np.random.seed(5)
x_reg = np.linspace(0, 1, 20)
y_reg = 0.5 + 0.8*x_reg - 0.3*x_reg**2 + 0.15*np.random.randn(20)

# Degree-7 polynomial features
def poly_features(x, d):
    return np.column_stack([x**i for i in range(d+1)])

X_reg = poly_features(x_reg, 7)
Y_reg = y_reg.reshape(-1,1)

def ridge_fit(X, Y, lam):
    n = X.shape[1]
    I = np.eye(n); I[0,0] = 0  # don't penalize bias
    return np.linalg.inv(X.T @ X + lam*I) @ X.T @ Y

x_plot = np.linspace(-0.1, 1.1, 200)
X_plot = poly_features(x_plot, 7)

lambdas = [0, 0.001, 0.01, 0.1, 1.0, 10.0]
colors = plt.cm.viridis(np.linspace(0, 1, len(lambdas)))

plt.figure(figsize=(10, 5))
plt.scatter(x_reg, y_reg, c='black', s=40, zorder=5, label='Training data')

for lam, color in zip(lambdas, colors):
    W_fit = ridge_fit(X_reg, Y_reg, lam)
    y_fit = X_plot @ W_fit
    plt.plot(x_plot, y_fit, color=color, linewidth=1.5, label=f'λ={lam}')

plt.ylim(-0.5, 2); plt.xlim(-0.1, 1.1)
plt.xlabel('x'); plt.ylabel('y')
plt.title('λ=0: overfits | λ large: underfits | λ moderate: best generalization')
plt.legend(bbox_to_anchor=(1.02, 1)); plt.tight_layout(); plt.show()

---
## Summary

**Bayesian vs Standard Regression:**

| | Standard (MLE) | Bayesian |
|---|---|---|
| Output | Single weight vector | Distribution over weights |
| Uncertainty | None | Explicit confidence intervals |
| Regularization | Added ad hoc | Comes naturally from prior |
| Computation | Gradient descent | Matrix inversion (small scale) |

**Key insight:** Regularization = Bayesian prior that weights are small. This is not just math — it reflects a real belief: in most problems, simpler explanations (smaller weights) generalize better.

**For the exam:** You don't implement Bayesian regression from scratch — you implement regularized regression. But knowing *why* the regularization term is there helps you understand what λ does and why we skip the bias term.

**Next:** [L09_classification.ipynb](L09_classification.ipynb) — sigmoid, logistic regression, and neural networks.